## Task 2: Data Cleaning & Preprocessing
Goal:
    Prepare the dataset for analysis by cleaning and organizing data.
Key Requirements:
    • Handle missing values
    • Remove duplicates
    • Format data correctly
Key Skills:
    Data preprocessing, data quality management

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings("ignore")

In [2]:
plt.rcParams.update({
    "figure.facecolor": "#f5f5f0",
    "axes.facecolor":   "#f5f5f0",
    "axes.edgecolor":   "#4a6741",
    "axes.labelcolor":  "#3a5232",
    "xtick.color":      "#3a5232",
    "ytick.color":      "#3a5232",
    "text.color":       "#3a5232",
    "font.family":      "DejaVu Sans",
    "axes.titlesize":   13,
    "axes.labelsize":   11,
})
GREEN  = "#4a6741"
TEAL   = "#5b9b8a"
ORANGE = "#e07b39"
PALETTE = [GREEN, ORANGE, TEAL, "#8fbc8f", "#d4a76a"]
sns.set_palette(PALETTE)

In [3]:
# ══════════════════════════════════════════════════════════
#  TASK 2 — DATA CLEANING & PREPROCESSING
# ══════════════════════════════════════════════════════════
print("\n" + "═"*60)
print("  TASK 2 — DATA CLEANING & PREPROCESSING")
print("═"*60)
df_raw = pd.read_csv("Titanic-Dataset.csv")
df = df_raw.copy()


════════════════════════════════════════════════════════════
  TASK 2 — DATA CLEANING & PREPROCESSING
════════════════════════════════════════════════════════════


In [4]:
# 1. Missing values BEFORE
print(f"\n Missing values BEFORE cleaning:")
print(df.isnull().sum()[df.isnull().sum() > 0])


 Missing values BEFORE cleaning:
Age         177
Cabin       687
Embarked      2
dtype: int64


In [5]:
# 2. Drop Cabin (>77% missing — not salvageable)
df.drop(columns=["Cabin"], inplace=True)
print("\nDropped 'Cabin' column (77% missing, not recoverable)")


Dropped 'Cabin' column (77% missing, not recoverable)


In [6]:
# 3. Fill Age with median (robust to outliers)
age_median = df["Age"].median()
df["Age"].fillna(age_median, inplace=True)
print(f"Filled {df_raw['Age'].isnull().sum()} missing 'Age' values with median ({age_median})")

Filled 177 missing 'Age' values with median (28.0)


In [7]:
# 4. Fill Embarked with mode (only 2 missing)
embarked_mode = df["Embarked"].mode()[0]
df["Embarked"].fillna(embarked_mode, inplace=True)
print(f"Filled 2 missing 'Embarked' values with mode ('{embarked_mode}')")

Filled 2 missing 'Embarked' values with mode ('S')


In [8]:
# 5. Remove duplicates
dupes_before = df.duplicated().sum()
df.drop_duplicates(inplace=True)
print(f"Duplicates removed: {dupes_before}")

Duplicates removed: 0


In [10]:
# 6. Feature engineering — extract Title from Name
df["Title"] = df["Name"].str.extract(r",\s*([^\.]+)\.")
df["Title"] = df["Title"].replace(
    ["Lady","Countess","Capt","Col","Don","Dr","Major","Rev","Sir","Jonkheer","Dona"],
    "Rare"
)
df["Title"] = df["Title"].replace({"Mlle":"Miss","Ms":"Miss","Mme":"Mrs"})
print("Extracted 'Title' feature from Name column")

Extracted 'Title' feature from Name column


In [9]:
# 7. Create FamilySize
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
print("Created 'FamilySize' = SibSp + Parch + 1")

Created 'FamilySize' = SibSp + Parch + 1


In [11]:
# 8. Create IsAlone
df["IsAlone"] = (df["FamilySize"] == 1).astype(int)
print("Created 'IsAlone' flag")

Created 'IsAlone' flag


In [12]:
# 9. Encode categoricals
df["Sex_enc"]      = LabelEncoder().fit_transform(df["Sex"])
df["Embarked_enc"] = LabelEncoder().fit_transform(df["Embarked"])
df["Title_enc"]    = LabelEncoder().fit_transform(df["Title"])
print("Label-encoded: Sex, Embarked, Title")

Label-encoded: Sex, Embarked, Title


In [13]:
# 10. Drop columns not needed for modelling
df.drop(columns=["PassengerId","Name","Ticket"], inplace=True)

In [14]:
print(f"\nMissing values AFTER cleaning: {df.isnull().sum().sum()} total")
print(f"Final dataset shape: {df.shape}")
print(f"\nSample cleaned rows:")
print(df.head(3).to_string())


Missing values AFTER cleaning: 0 total
Final dataset shape: (891, 14)

Sample cleaned rows:
   Survived  Pclass     Sex   Age  SibSp  Parch     Fare Embarked  FamilySize Title  IsAlone  Sex_enc  Embarked_enc  Title_enc
0         0       3    male  22.0      1      0   7.2500        S           2    Mr        0        1             2          2
1         1       1  female  38.0      1      0  71.2833        C           2   Mrs        0        0             0          3
2         1       3  female  26.0      0      0   7.9250        S           1  Miss        1        0             2          1
